In [1]:
import pandas as pd
import numpy as np
import os
import random
from sklearn.metrics import mean_squared_error
import matplotlib.pyplot as plt

In [ ]:
data = ['coin']
model_name = 'trTFTF'
num = 100  # Model k per loss

os.makedirs("inference", exist_ok = True)

In [ ]:
for d in range(len(data)):
    #############################################################################################
    target_X = pd.read_csv(f"../data/{data[d]}/train_input_7.csv").iloc[:, 1:].values.astype(np.float32)
    target_y = pd.read_csv(f"../data/{data[d]}/train_output_7.csv").iloc[:, 1:].values.astype(np.float32)

    target_X_val = target_X[-round(target_X.shape[0] * 0.2):, :].astype(np.float32)
    target_y_val = target_y[-round(target_y.shape[0] * 0.2):].astype(np.float32)

    target_X = target_X[:-round(target_X.shape[0] * 0.2), :].astype(np.float32)
    target_y = target_y[:-round(target_y.shape[0] * 0.2)].astype(np.float32)
    test_X = pd.read_csv(f"../data/{data[d]}/val_input_7.csv").iloc[:, 1:].values.astype(np.float32)
    test_y = pd.read_csv(f"../data/{data[d]}/val_output_7.csv").iloc[:, 1:].values.astype(np.float32)

    #############################################################################################
    folder_path = f'result/{data[d]}/test/'
    folder_path2 = f'result/{data[d]}/val/'

    all_files = os.listdir(folder_path)
    all_files2 = os.listdir(folder_path2)
    print(f"--- 파일 검색 디버깅 ---")
    print(f"검색 중인 폴더: {folder_path}")
    print(f"폴더 안의 모든 파일: {all_files}")
    search_prefix = f"{model_name}_{data[d]}"
    print(f"찾고 있는 파일 시작 부분(prefix): '{search_prefix}'")
    print("--- 디버깅 끝 ---")
    files = sorted([f for f in all_files if f.startswith(f"{model_name}_{data[d]}") and f.endswith("pred.csv")])
    dataframes = [pd.read_csv(os.path.join(folder_path, f)) for f in files]

    files2 = sorted([f for f in all_files2 if f.startswith(f"{model_name}_{data[d]}") and f.endswith("pred.csv")])
    dataframes2 = [pd.read_csv(os.path.join(folder_path2, f)) for f in files2]

    all = np.array([np.stack(dataframes[i].iloc[:, 1:].values.reshape(num, -1, target_y.shape[1])) for i in range(len(dataframes))])
    all_val = np.array([np.stack(dataframes2[i].iloc[:, 1:].values.reshape(num, -1, target_y.shape[1])) for i in range(len(dataframes2))])
    #############################################################################################
    all_ens_lst, all_ens_rmse_lst = [], []
    bolt1_lst, bolt1_rmse_lst = [], []
    bolt3_lst, bolt3_rmse_lst = [], []
    bolt5_lst, bolt5_rmse_lst = [], []
    
    p = 10
    b = 50
    np.random.seed(100)  ### NEW: mean 평가 지표 기록용

    for i in range(b):
        # --------------------------------------------------------------
        # 1) 무작위 인덱스 샘플링
        # matrix = np.arange(100).reshape(10, 10)
        nums1 = random.sample(range(num), k=p)
        nums2 = random.sample(range(num), k=p)
        nums3 = random.sample(range(num), k=p)
        nums4 = random.sample(range(num), k=p)
        nums5 = random.sample(range(num), k=p)

        # nums1 = matrix[i]
        # nums2 = matrix[i]
        # nums3 = matrix[i]
        # nums4 = matrix[i]
        # nums5 = matrix[i]
        # --------------------------------------------------------------
        # 2) all_Ens (median 앙상블은 그대로)
        score = np.concatenate([all[0][nums1], all[1][nums2],
                                all[2][nums3], all[3][nums4],
                                all[4][nums5]], axis=0)

        all_ens = np.median(score, axis=0).flatten()
        all_ens_rmse = np.sqrt(mean_squared_error(test_y.flatten(), all_ens))
        all_ens_lst.append(all_ens)
        all_ens_rmse_lst.append(all_ens_rmse)

        ####################################################################
        mae_val = np.median(all_val[0][nums1], axis=0).flatten()  # .shape
        mape_val = np.median(all_val[1][nums2], axis=0).flatten()
        mase_val = np.median(all_val[2][nums3], axis=0).flatten()
        mse_val = np.median(all_val[3][nums4], axis=0).flatten()
        smape_val = np.median(all_val[4][nums5], axis=0).flatten()

        rmse_mae_val = np.sqrt(mean_squared_error(target_y_val.flatten(), mae_val))
        rmse_mape_val = np.sqrt(mean_squared_error(target_y_val.flatten(), mape_val))
        rmse_mase_val = np.sqrt(mean_squared_error(target_y_val.flatten(), mase_val))
        rmse_mse_val = np.sqrt(mean_squared_error(target_y_val.flatten(), mse_val))
        rmse_smape_val = np.sqrt(mean_squared_error(target_y_val.flatten(), smape_val))

        fin_pred_mae = np.median(all[0][nums1], axis=0).flatten()
        fin_pred_mape = np.median(all[1][nums2], axis=0).flatten()
        fin_pred_mase = np.median(all[2][nums3], axis=0).flatten()
        fin_pred_mse = np.median(all[3][nums4], axis=0).flatten()
        fin_pred_smape = np.median(all[4][nums5], axis=0).flatten()

        performance = np.array([rmse_mae_val, rmse_mape_val, rmse_mase_val, rmse_mse_val, rmse_smape_val])

        for beta in [1, 3, 5]:
            bolt_rmse_lst_ = []
            bolt = []
            
            weights = np.exp(-beta * performance)
            gd = np.concatenate([fin_pred_mae.flatten().reshape(1, -1),
                                 fin_pred_mape.flatten().reshape(1, -1),
                                 fin_pred_mase.flatten().reshape(1, -1),
                                 fin_pred_mse.flatten().reshape(1, -1),
                                 fin_pred_smape.flatten().reshape(1, -1)], axis=0)

            normalized_weights = weights / np.sum(weights)

            # 각 모델의 예측값에 가중치를 부여하여 앙상블 예측 생성
            ensemble_prediction = np.dot(normalized_weights, gd)
            bolt.append(ensemble_prediction)
            bolt_rmse = np.sqrt(mean_squared_error(test_y.flatten(), ensemble_prediction.flatten()))
            bolt_rmse_lst_.append(bolt_rmse)

            if beta == 1:
                bolt1_lst.append(bolt)
                bolt1_rmse_lst.append(bolt_rmse_lst_)
            elif beta == 3:
                bolt3_lst.append(bolt)
                bolt3_rmse_lst.append(bolt_rmse_lst_)
            else:
                bolt5_lst.append(bolt)
                bolt5_rmse_lst.append(bolt_rmse_lst_)

        # --------------------------------------------------------------

    a1 = np.min(np.array(bolt1_rmse_lst), axis=0)
    a3 = np.min(np.array(bolt3_rmse_lst), axis=0)
    a5 = np.min(np.array(bolt5_rmse_lst), axis=0)

    summary = pd.DataFrame({'All Ens': all_ens_rmse_lst,
                            'Exp(1)': np.array(bolt1_rmse_lst).T[np.argmin(a1)],
                            'Exp(3)': np.array(bolt3_rmse_lst).T[np.argmin(a3)],
                            'Exp(5)': np.array(bolt5_rmse_lst).T[np.argmin(a5)]})
    summary.to_csv(f'inference/{model_name}_{data[d]}_summary.csv')
    summary.describe()

--- 파일 검색 디버깅 ---
검색 중인 폴더: result/coin/test/
폴더 안의 모든 파일: ['trTFMLP_coin_mse_pred.csv', 'trTFMLP_coin_mae_pred.csv', 'trTFMLP_coin_smape_pred.csv', 'trTFMLP_coin_mase_pred.csv', 'trTFTF_coin_SMAPE_pred.csv', 'trTFTF_coin_mse_pred.csv', 'trTFTF_coin_mape_pred.csv', 'trTFTF_coin_MASE_pred.csv', 'trTFMLP_coin_mape_pred.csv', 'trTFTF_coin_mae_pred.csv']
찾고 있는 파일 시작 부분(prefix): 'trTFTF_coin'
--- 디버깅 끝 ---


In [ ]:
rec = []
for i in range(len(data)):
    d= pd.read_csv(f'inference/{model_name}_{data[i]}_summary.csv').iloc[:,1:].describe().T.iloc[:,1:3].round(5)
    d['mean(std)'] = d.apply(
    lambda r: f"{r['mean']:.5f} ({r['std']:.5f})", axis=1)
    
    rec.append(d.loc[:,'mean(std)'])

dfff = pd.DataFrame(rec).T

dfff.columns = data
dfff = dfff.T.reset_index().rename({"index": "dataset"}, axis = 1).assign(model = model_name).set_index("model").reset_index()

,model,dataset,BOLT1,BOLT3,BOLT5,Median
0,trTFTF,coin,3.84203 (0.01124),3.82882 (0.01135),3.82429 (0.01217),3.85886 (0.01547)


In [18]:
pd.DataFrame(rec)

,BOLT1,BOLT3,BOLT5,Median
mean(std),3.84203 (0.01124),3.82882 (0.01135),3.82429 (0.01217),3.85886 (0.01547)
